In [4]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import time
import random

# Custom headers to mimic a browser
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
}

# Retry logic for requests

def fetch_with_retries(url, max_retries=3, backoff=2):
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            if response.status_code == 200:
                return response
            else:
                print(f"Non-200 status code: {response.status_code}")
        except Exception as e:
            print(f"Request failed (attempt {attempt+1}): {e}")
        time.sleep(backoff ** attempt + random.uniform(0, 1))
    return None

# URL for Bol News Sports
news_url = "https://www.bolnews.com/sports/"

response = fetch_with_retries(news_url)
if not response:
    print("Failed to fetch the main page after retries.")
else:
    news_soup = BeautifulSoup(response.content, "html.parser")
    # Find all news cards using the updated selector
    articles_container = news_soup.select_one("body > div.main-wrapper > section:nth-child(3)")
    articles = []
    if articles_container:
        articles = articles_container.find_all("div", recursive=False)
    if not articles:
        print("No articles found using the provided selector. Trying alternative selector...")
        articles = news_soup.find_all("article")
    if not articles:
        print("No articles found.")
    else:
        for idx, article in enumerate(articles, start=1):
            # Extract headline
            headline_tag = article.find(['h2', 'h3', 'h4'])
            headline = headline_tag.get_text(strip=True) if headline_tag else '(No headline)'
            # Extract link
            link_tag = article.find('a', href=True)
            link = link_tag['href'] if link_tag else None
            if link and link.startswith("/"):
                link = "https://www.bolnews.com" + link
            print(f"{idx}. {headline}")
            print(f"   Link: {link if link else '(No link found)'}")
            # Fetch article details with delay and retry
            snippet = ''
            date_str = ''
            if link:
                time.sleep(random.uniform(2, 5))  # Random delay
                article_resp = fetch_with_retries(link)
                if article_resp:
                    article_soup = BeautifulSoup(article_resp.content, "html.parser")
                    # Extract paragraphs for snippet
                    paragraphs = article_soup.find_all("p")
                    article_text = " ".join([p.get_text(strip=True) for p in paragraphs])
                    snippet = article_text[:400] + ("..." if len(article_text) > 400 else "")
                    # Extract date
                    time_tag = article_soup.find("time")
                    if not time_tag:
                        meta_time = article_soup.find("meta", attrs={"property": "article:published_time"})
                        if meta_time and meta_time.has_attr("content"):
                            date_str = meta_time["content"]
                    if not date_str and time_tag and time_tag.has_attr("datetime"):
                        date_str = time_tag["datetime"]
                    elif not date_str and time_tag:
                        date_str = time_tag.get_text(strip=True)
                    if date_str:
                        try:
                            dt = datetime.fromisoformat(date_str.replace("Z", "+00:00"))
                            date_str = dt.strftime("%Y-%m-%d %H:%M:%S %Z")
                        except Exception:
                            pass
                else:
                    snippet = "(Could not fetch article)"
                    date_str = ''
            print(f"   Date: {date_str if date_str else '(No date found)'}")
            print(f"   News: {snippet}")

1. Football News
   Link: https://www.bolnews.com/sports/2025/05/india-blocks-social-media-accounts-of-babar-azam-other-pakistani-cricketers/
   Date: 2025-05-02 17:33:40 UTC
   News: Follow us on loading.... loading.... loading.... loading.... loading.... loading.... loading.... 02nd May, 2025. 10:33 pm Share Listen Font size Dark Mode Save Print India blocks social media accounts of Babar Azam, other Pakistani cricketers TheIndian governmentblocked the social media accounts of several Pakistani cricketers, including Babar Azam and Mohammad Rizwan, following the Pahalgam attac...
   Date: 2025-05-02 17:33:40 UTC
   News: Follow us on loading.... loading.... loading.... loading.... loading.... loading.... loading.... 02nd May, 2025. 10:33 pm Share Listen Font size Dark Mode Save Print India blocks social media accounts of Babar Azam, other Pakistani cricketers TheIndian governmentblocked the social media accounts of several Pakistani cricketers, including Babar Azam and Mohammad Rizwan

In [6]:
import requests
from bs4 import BeautifulSoup
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

def get_bolnews_headlines():
    """Extract headlines from Bol News main page using robust methods"""
    url = "https://www.bolnews.com/"
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            print(f"Error: Received status code {response.status_code}")
            return []
        soup = BeautifulSoup(response.content, "html.parser")
        headlines = []
        seen_urls = set()
        # Strategy 1: Find headline containers directly (Bol News uses h2/h3/h4 for headlines)
        for container in soup.select('h2, h3, h4'):
            link_tag = container.find('a', href=True) or container.find_parent('a', href=True)
            text = container.get_text(strip=True)
            link = None
            if link_tag:
                link = link_tag.get('href', '')
            # Only add if text and link are valid and not already seen
            if text and link and link not in seen_urls:
                if link.startswith('/'):
                    link = f"https://www.bolnews.com{link}"
                headlines.append((text, link))
                seen_urls.add(link)
        # Strategy 2: Find all article links by their URL pattern
        if len(headlines) < 5:
            for link_tag in soup.find_all('a', href=True):
                link = link_tag.get('href', '')
                text = link_tag.get_text(strip=True)
                if (text and link and link not in seen_urls and
                    ('/news/' in link or '/sports/' in link or '/latest/' in link) and
                    not link.endswith('.jpg') and not link.endswith('.png')):
                    if link.startswith('/'):
                        link = f"https://www.bolnews.com{link}"
                    headlines.append((text, link))
                    seen_urls.add(link)
        return headlines[:10]
    except Exception as e:
        print(f"Error scraping Bol News: {e}")
        return []

print("\nBol News Headlines:")
for idx, (text, link) in enumerate(get_bolnews_headlines(), 1):
    print(f"{idx}. {text}\n   {link}")


Bol News Headlines:
1. Sports
   https://www.bolnews.com/./sports/
2. Cricket
   https://www.bolnews.com/sports/cricket/
3. Hockey
   https://www.bolnews.com/sports/hockey/
4. Football
   https://www.bolnews.com/sports/football/
5. Squash
   https://www.bolnews.com/sports/squash/
6. Snooker
   https://www.bolnews.com/sports/snooker/
7. Tennis
   https://www.bolnews.com/sports/tennis/
8. Atheletics
   https://www.bolnews.com/sports/atheletics/
9. Shooting
   https://www.bolnews.com/sports/shooting
10. Volleyball
   https://www.bolnews.com/sports/volleyball
1. Sports
   https://www.bolnews.com/./sports/
2. Cricket
   https://www.bolnews.com/sports/cricket/
3. Hockey
   https://www.bolnews.com/sports/hockey/
4. Football
   https://www.bolnews.com/sports/football/
5. Squash
   https://www.bolnews.com/sports/squash/
6. Snooker
   https://www.bolnews.com/sports/snooker/
7. Tennis
   https://www.bolnews.com/sports/tennis/
8. Atheletics
   https://www.bolnews.com/sports/atheletics/
9. Shootin

In [ ]:
import requests
from bs4 import BeautifulSoup
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

def get_bolnews_content_previews():
    """Extract content previews from Bol News main page."""
    url = "https://www.bolnews.com/"
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            print(f"Error: Received status code {response.status_code}")
            return []
        soup = BeautifulSoup(response.content, "html.parser")
        previews = []
        seen_urls = set()
        # Find all news cards or main content blocks
        for card in soup.select('div.news-card, div[class*="post"], article'):
            # Headline
            headline_tag = card.find(['h2', 'h3', 'h4'])
            headline = headline_tag.get_text(strip=True) if headline_tag else None
            # Link
            link_tag = card.find('a', href=True)
            link = link_tag['href'] if link_tag else None
            if link and link.startswith('/'):
                link = f"https://www.bolnews.com{link}"
            # Preview/snippet
            preview_tag = card.find('p')
            preview = preview_tag.get_text(strip=True) if preview_tag else None
            if headline and link and preview and link not in seen_urls:
                previews.append((headline, link, preview))
                seen_urls.add(link)
        # Fallback: Try to get previews from all links if not enough found
        if len(previews) < 5:
            for link_tag in soup.find_all('a', href=True):
                link = link_tag.get('href', '')
                text = link_tag.get_text(strip=True)
                if (text and link and link not in seen_urls and
                    ('/news/' in link or '/sports/' in link or '/latest/' in link) and
                    not link.endswith('.jpg') and not link.endswith('.png')):
                    # Try to get a preview from a nearby <p> tag
                    preview = None
                    parent = link_tag.find_parent(['div', 'article'])
                    if parent:
                        preview_tag = parent.find('p')
                        if preview_tag:
                            preview = preview_tag.get_text(strip=True)
                    if link.startswith('/'):
                        link = f"https://www.bolnews.com{link}"
                    if preview:
                        previews.append((text, link, preview))
                        seen_urls.add(link)
        return previews[:10]
    except Exception as e:
        print(f"Error scraping Bol News: {e}")
        return []

print("\nBol News Content Previews:")
for idx, (headline, link, preview) in enumerate(get_bolnews_content_previews(), 1):
    print(f"{idx}. {headline}\n   {link}\n   Preview: {preview}\n")